In [0]:
import dlt
from pyspark.sql.functions import *

base_path = "/Volumes/DLT_HC_Fraud_detection/bronze_silver_gold/healthcare_fraud_dlt"

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

# base_path = "/Volumes/DLT_HC_Fraud_detection/bronze_silver_gold/healthcare_fraud_dlt"

# ---------------- JSON SCHEMAS ----------------
rules_schema = StructType([
    StructField("rule_id", IntegerType(), True),
    StructField("rule", StringType(), True)
])

audit_schema = StructType([
    StructField("log_id", IntegerType(), True),
    StructField("claim_id", IntegerType(), True),
    StructField("event", StringType(), True),
    StructField("timestamp", StringType(), True)
])

claim_lines_schema = StructType([
    StructField("claim_id", IntegerType(), True),
    StructField("procedures", ArrayType(
        StructType([
            StructField("code", StringType(), True),
            StructField("charge", DoubleType(), True)
        ])
    ), True)
])

# ---------------- READERS ----------------
def read_csv(path):
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("header","true")
        .option("inferSchema","true")
        .load(path)
        .withColumn("ingestion_ts", current_timestamp())
    )

def read_json(path, schema):
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format","json")
        .schema(schema)
        .load(path)
        .withColumn("ingestion_ts", current_timestamp())
    )

# ---------------- CSV TABLES ----------------
@dlt.table
def bronze_claims():
    return read_csv(f"{base_path}/HC_2026_claims")

@dlt.table
def bronze_patients():
    return read_csv(f"{base_path}/HC_2026_patients")

@dlt.table
def bronze_payments():
    return read_csv(f"{base_path}/HC_2026_payments")

@dlt.table
def bronze_providers():
    return read_csv(f"{base_path}/HC_2026_providers")

# ---------------- JSON TABLES ----------------
@dlt.table
def bronze_claim_lines():
    return read_json(f"{base_path}/HC_2026_claim_lines", claim_lines_schema)

@dlt.table
def bronze_rules():
    return read_json(f"{base_path}/HC_2026_rules", rules_schema)

@dlt.table
def bronze_audit_logs():
    return read_json(f"{base_path}/HC_2026_audit_logs", audit_schema)

In [0]:
import dlt
from pyspark.sql.functions import *

# ---------- HELPER: SAFE DATE PARSER ----------
def parse_date(c):
    return coalesce(
        to_date(c, "dd-MM-yyyy"),
        to_date(c, "yyyy-MM-dd"),
        to_date(c, "MM/dd/yyyy")
    )

# ---------- CLAIMS ----------
@dlt.table
def silver_claims():
    df = dlt.read("bronze_claims")

    df = df.select(
        col("claim_id").cast("int"),
        col("provider_id").cast("int"),
        col("patient_id").cast("int"),
        col("claim_amount").cast("double"),
        parse_date(col("claim_date")).alias("claim_date"),
        col("flagged").cast("int"),
        "ingestion_ts"
    )

    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    return df.dropDuplicates(["claim_id","ingestion_ts"])


# ---------- PROVIDERS ----------
@dlt.table
def silver_providers():
    df = dlt.read("bronze_providers")

    df = df.select(
        col("provider_id").cast("int"),
        col("name"),
        col("specialty"),
        col("risk_score").cast("double"),
        "ingestion_ts"
    )

    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    return df.dropDuplicates(["provider_id","ingestion_ts"])


# ---------- PATIENTS ----------
@dlt.table
def silver_patients():
    df = dlt.read("bronze_patients")

    df = df.select(
        col("patient_id").cast("int"),
        col("age").cast("int"),
        col("gender"),
        "ingestion_ts"
    )

    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    return df.dropDuplicates(["patient_id","ingestion_ts"])


# ---------- PAYMENTS (AGGREGATED PER CLAIM) ----------
@dlt.table
def silver_payments():
    df = dlt.read("bronze_payments")

    df = df.select(
        col("claim_id").cast("int"),
        col("paid_amount").cast("double")
    )

    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    # Aggregate so Gold joins don't duplicate rows
    return df.groupBy("claim_id").agg(sum("paid_amount").alias("paid_amount"))


# ---------- CLAIM LINES ----------
@dlt.table
def silver_claim_lines():
    df = dlt.read("bronze_claim_lines")

    df = df.select(
        col("claim_id").cast("int"),
        explode("procedures").alias("proc"),
        "ingestion_ts"
    ).select(
        "claim_id",
        col("proc.code").alias("procedure_code"),
        col("proc.charge").cast("double").alias("charge"),
        "ingestion_ts"
    )

    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    return df


# ---------- RULES ----------
@dlt.table
def silver_rules():
    df = dlt.read("bronze_rules")

    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    return df


# ---------- AUDIT LOGS ----------
@dlt.table
def silver_audit_logs():
    df = dlt.read("bronze_audit_logs")

    df = df.withColumn("log_ts", to_timestamp("timestamp")).drop("timestamp")

    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")

    return df

In [0]:
import dlt
from pyspark.sql.functions import *

# ============================================================
# GOLD FRAUD SIGNALS — FINAL VERSION (STRICT PAYMENT RULE)
# ============================================================
@dlt.table(comment="Fraud detection output with strict payment validation")
def gold_fraud_signals():

    # -----------------------------
    # READ SILVER
    # -----------------------------
    claims = dlt.read("silver_claims").alias("c")
    providers = dlt.read("silver_providers").alias("p")
    payments = dlt.read("silver_payments").alias("pay")

    # -----------------------------
    # DUPLICATE DETECTION
    # -----------------------------
    dup_df = (
        claims.groupBy("claim_id")
        .agg(count("*").alias("cnt"))
        .filter(col("cnt") > 1)
        .select(col("claim_id").alias("dup_claim_id"))
        .withColumn("dup_flag", lit(1))
    )

    # -----------------------------
    # PROVIDER PEER AVERAGE
    # -----------------------------
    peer_avg = (
        claims.groupBy("provider_id")
        .agg(avg("claim_amount").alias("provider_avg_claim"))
    )

    # -----------------------------
    # JOIN
    # -----------------------------
    df = (
        claims.join(providers, "provider_id", "left")
              .join(payments, "claim_id", "left")
              .join(dup_df, claims.claim_id == dup_df.dup_claim_id, "left")
              .join(peer_avg, "provider_id", "left")
    )

    df = (
        df.withColumn("paid_amount", coalesce(col("paid_amount"), lit(0)))
          .withColumn("dup_flag", coalesce(col("dup_flag"), lit(0)))
          .withColumn("provider_avg_claim", coalesce(col("provider_avg_claim"), lit(0)))
          .withColumn("risk_score", coalesce(col("risk_score"), lit(0)))
    )

    # =====================================================
    # PROVIDER RISK LEVEL
    # =====================================================
    df = df.withColumn(
        "provider_risk_level",
        when(col("risk_score") >= 90,"HIGH")
        .when(col("risk_score") >= 70,"MEDIUM")
        .otherwise("LOW")
    )

    # =====================================================
    # FRAUD FLAGS (NUMERIC)
    # =====================================================

    # Duplicate
    df = df.withColumn("duplicate_flag", col("dup_flag"))

    # High claim
    df = df.withColumn(
        "high_claim_flag",
        when(col("claim_amount") >= 75000,1).otherwise(0)
    )

    # 🔴 FIXED PAYMENT RULE HERE
    df = df.withColumn(
        "payment_anomaly_flag_num",
        when(col("paid_amount") > col("claim_amount"),1).otherwise(0)
    )

    # Peer deviation
    df = df.withColumn(
        "peer_deviation_flag",
        when(col("claim_amount") > col("provider_avg_claim") * 2,1).otherwise(0)
    )

    # =====================================================
    # FRAUD SCORE
    # =====================================================
    df = df.withColumn(
        "fraud_score",
        col("duplicate_flag")*40 +
        col("payment_anomaly_flag_num")*30 +
        col("high_claim_flag")*20 +
        col("peer_deviation_flag")*20 +
        when(col("provider_risk_level")=="MEDIUM",15).otherwise(0) +
        when(col("provider_risk_level")=="HIGH",30).otherwise(0)
    )

    # =====================================================
    # TRIGGER SOURCE
    # =====================================================
    df = df.withColumn(
        "provider_trigger",
        when(col("provider_risk_level").isin("MEDIUM","HIGH"),1).otherwise(0)
    )

    df = df.withColumn(
        "claim_trigger",
        when(
            (col("duplicate_flag")+
             col("payment_anomaly_flag_num")+
             col("high_claim_flag")+
             col("peer_deviation_flag")) > 0,1
        ).otherwise(0)
    )

    df = df.withColumn(
        "fraud_trigger_source",
        when((col("provider_trigger")==1) & (col("claim_trigger")==1),"BOTH")
        .when(col("provider_trigger")==1,"PROVIDER_RISK")
        .when(col("claim_trigger")==1,"CLAIM_ANOMALY")
        .otherwise("NONE")
    )

    # =====================================================
    # FINAL DECISION
    # =====================================================
    df = df.withColumn(
        "fraud_flag",
        when(col("fraud_score") >= 60,"YES")
        .when(col("fraud_score") >= 30,"REVIEW")
        .otherwise("NO")
    )

    df = df.withColumn(
        "fraud_severity",
        when(col("fraud_score") >= 80,"HIGH")
        .when(col("fraud_score") >= 50,"MEDIUM")
        .otherwise("LOW")
    )

    # =====================================================
    # BUSINESS STATUS + REASONS
    # =====================================================
    df = df.withColumn(
        "payment_integrity_status",
        when(col("payment_anomaly_flag_num")==1,"TRIGGERED")
        .otherwise("NORMAL")
    ).withColumn(
        "payment_integrity_reason",
        when(col("payment_anomaly_flag_num")==1,
             "Payment exceeds billed claim amount")
        .otherwise("Payment aligned with billed amount")
    )

    # =====================================================
    # FINAL PROJECTION
    # =====================================================
    df = df.select(
        col("claim_id"),
        col("provider_id"),
        col("patient_id"),
        col("claim_amount"),
        col("paid_amount"),
        col("claim_date"),
        col("name").alias("provider_name"),
        col("specialty"),
        col("provider_risk_level"),
        col("provider_avg_claim"),
        col("payment_integrity_status"),
        col("payment_integrity_reason"),
        col("fraud_score"),
        col("fraud_trigger_source"),
        col("fraud_flag"),
        col("fraud_severity"),
        col("c.ingestion_ts").alias("ingestion_ts")
    )

    return df